### dbGap FHIR Notebooks - Exercise 3

## Learning Objectives and Key Concepts

In this exercise, you will query the dbGaP FHIR test study: phs002409: 


- Recreate subject data as submitted 
- Aggregate FHIR Observations for a subject
- Enable Dataframe queries 
- Explore the same queries via FHIR

## Motivation/Purpose
To explore challenges in different representations of dbGaP data

## Requires
- ipywidgets
- pandas

 ### Icons in this Guide
 📘 A link to a useful external reference related to the section the icon appears in  

 🖐 A hands-on section where you will code something or interact with the server  
 
 
Acknowledging use of code snippets from [NIH FHIR training](https://github.com/NIH-ODSS/fhir-exercises/tree/main/Python) Exercise 0.



In [1]:
from dbgap_fhir import DbGapFHIR

FHIR_SERVER = "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1"
mf = DbGapFHIR(FHIR_SERVER)

### Query subjects for the study

First some basic exploration of the data in the study is helpful via FHIR.

Find all the patients registered as subjects to the study.

In [2]:
study_id = "phs002409"
patients = mf.run_query(
    f"Patient?_has:ResearchSubject:individual:study={study_id}"
)

Total  Resources: 813
Total  Bytes: 365840
Total  Pages: 9
Time elapsed 8.2713 seconds


In [3]:
patients[812]

{'resourceType': 'Patient',
 'id': '3635999',
 'meta': {'tag': [{'system': 'https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/DbGaP-Backend-Source',
    'code': 'Nitro',
    'display': 'Resource served by Nitro backend.'}]},
 'identifier': [{'id': '3635999',
   'system': 'https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/DbGaPConcept-DbGaPSubjectIdentifier'}],
 'gender': 'unknown'}

### Look at the study details

In [4]:
studies = mf.run_query(f"ResearchStudy?_id={study_id}")

Total  Resources: 1
Total  Bytes: 8030
Total  Pages: 1
Time elapsed 0.0347 seconds


In [5]:
studies[0]

{'resourceType': 'ResearchStudy',
 'id': 'phs002409',
 'meta': {'versionId': '2',
  'lastUpdated': '2024-04-22T21:05:52.544-04:00',
  'source': '#Ko09D1iDEn7LILOH',
  'security': [{'system': 'https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/DbGaPConcept-SecurityStudyConsent',
    'code': 'public',
    'display': 'public'},
   {'system': 'http://terminology.hl7.org/CodeSystem/v3-Confidentiality',
    'code': 'U',
    'display': 'unrestricted'}]},
 'extension': [{'url': 'https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-StudyOverviewUrl',
   'valueUrl': 'https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id=phs002409.v1.p1'},
  {'url': 'https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-ReleaseDate',
   'valueDate': '2021-09-09'},
  {'url': 'https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-StudyConsents',
   'extension': [{'url': 'https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefini

### Observations
The query below was simply copied from the Kids First examples. It seems reasonable but is not yet implemented in dbGaP FHIR.

Patient?_has:ResearchSubject:individual:study={study_id}&_revinclude=Observation:subject"

We use instead a workaround to run a query for the Observations for each Patient.

The following creates a dataframe showing the observations for each patient.
It also
* saves the definition of each observation (column) to a file
* checks that the definition is the same for each instance of the observation are the same.

This is also an opportunity to illustrate the use of iPython widgets to display a progress bar.

📘 You can find out more about progress bars and other iPython widgets [here](https://ipywidgets.readthedocs.io/en/stable/).

Because we are going to make a large number of requests in a short period of time this is a good example of a task where having an api key is helpful.

📘See notebook 2 for details.

In [6]:
import os

API_KEY_PATH = "~/.keys/ncbi_api_key.txt"

# the os.path.expanduser expands file paths which include ~/ representation for the user's home directory
with open(os.path.expanduser(API_KEY_PATH)) as f:
    api_key = f.read()

    mf = DbGapFHIR(FHIR_SERVER, api_key=api_key)

### Now we can run the query
In fact one query per patient. It will take some time. The progress bar will do what progress bars do.

In [7]:
from ipywidgets import IntProgress
from IPython.display import display

prog = IntProgress(min=0, max=len(patients))  # instantiate the bar
display(prog)  # display the bar

all_obs = []
patients_with_obs = []
for p in patients:
    # print(p['id'])
    obs = mf.run_query(f"Observation?subject={p['id']}", show_stats=False)
    if len(obs) > 0:
        all_obs += obs
        patients_with_obs.append(p)
    prog.value += 1

print(f"{len(patients_with_obs)} patients had observations")
print(f"{len(all_obs)} total observations")

IntProgress(value=0, max=813)

813 patients had observations
41224 total observations


### Construct a dataframe

In [8]:
import json
import pandas as pd
from collections import Counter


patient_observations_dict = {}
variable_definitions = {}
observations = []
obsCounter = Counter()
codeCounter = Counter()
vccCounter = Counter()
printObsCounts = True
rlimit = 20
nn = 0
for r in all_obs:

    if r["resourceType"] == "Observation":
        # print(json.dumps(r,indent=3))
        # nn+=1
        # if nn > rlimit:
        #    break
        subject_id = r["subject"]["reference"]
        obsCounter[subject_id] += 1
        obs_display_name = r["code"]["coding"][0]["display"]
        if "valueQuantity" in r:
            value_text = r["valueQuantity"]["value"]
            # value_unit = r['valueQuantity']['unit']
        elif "valueCodeableConcept" in r:
            value_text = r["valueCodeableConcept"]["coding"][0]["display"]
        else:
            value_text = "unknown"
        codeCounter[obs_display_name] += 1
        # vccCounter[vcc_text] +=1
        observations.append(r)

        if subject_id not in patient_observations_dict:
            patient_observations_dict[subject_id] = {
                obs_display_name: value_text
            }
        else:
            patient_observations_dict[subject_id][obs_display_name] = value_text

# Summarize
print(f"Number of patients with observations {len(obsCounter.keys())}")

if printObsCounts:
    print("Observation count per patient")
    print(json.dumps(obsCounter, indent=3))
# print("Coding counts")
# print(json.dumps(codeCounter, indent=3))
df = pd.DataFrame.from_dict(codeCounter, orient="index")

Number of patients with observations 813
Observation count per patient
{
   "Patient/3635187": 49,
   "Patient/3635188": 52,
   "Patient/3635189": 50,
   "Patient/3635190": 51,
   "Patient/3635191": 51,
   "Patient/3635192": 50,
   "Patient/3635193": 52,
   "Patient/3635194": 51,
   "Patient/3635195": 51,
   "Patient/3635196": 50,
   "Patient/3635197": 50,
   "Patient/3635198": 51,
   "Patient/3635199": 51,
   "Patient/3635200": 52,
   "Patient/3635201": 51,
   "Patient/3635202": 51,
   "Patient/3635203": 51,
   "Patient/3635204": 52,
   "Patient/3635205": 51,
   "Patient/3635206": 49,
   "Patient/3635207": 48,
   "Patient/3635208": 50,
   "Patient/3635209": 51,
   "Patient/3635210": 51,
   "Patient/3635211": 51,
   "Patient/3635212": 48,
   "Patient/3635213": 52,
   "Patient/3635214": 51,
   "Patient/3635215": 49,
   "Patient/3635216": 51,
   "Patient/3635217": 51,
   "Patient/3635218": 50,
   "Patient/3635219": 50,
   "Patient/3635220": 50,
   "Patient/3635221": 50,
   "Patient/36352

### Create and display the Dataframe

In [9]:
pd.set_option("display.max_rows", 30, "display.max_columns", None)
patient_df = pd.DataFrame.from_dict(patient_observations_dict, orient="index")
# patient_df.fillna('', inplace=True)
display(patient_df)

,subj_id,consent_group,Gender,protocol,base_vnum,case_control,case_control_peak,Race - 2 character code PhenX,age_baseline,age_onset,Body height stand,Weight,Growth Chart Current BMI,bmiz_baseline,tx_grp,cs_tx,cs_comb_tx,clinic_city,smoke,ENV_SMOKE,ENV_SMOKE_pretrial,IUS,pred_bursts_event1_month,pred_bursts_event1_week,pred_bursts_event2_month,pred_bursts_event2_week,edhos_event1_month,edhos_event1_week,edhos_event2_month,edhos_event2_week,prech_long,prech_short,PC20_baseline,LNPC20_baseline,FEV1,FVC Vol Respiratory Spirometry,FEV1/FVC Predicted,PREFEVPP_baseline,PREFVCPP_baseline,bdrabpct_baseline,ATOPY,CompleteBloodCount Eosinophils Number,log10eos_baseline,IgE Concentration,ampf_baseline,totres_baseline,POS_SKINTEST_baseline,tot_pred_bursts,tot_edhos,amsym_baseline,cs_dose
Patient/3635187,10100288,1,2,101,1,2,2,1,7.2,2.0,135.5,34.6,15.02,1.17,9,0,0,2,0,0,0,0,8,17,48,17,48,207,48,207,21.60,24.29,0.198,2.13,1.24,2.85,91.0,87.0,92.0,8.0,0,900.0,2.710117,2.08,265.714286,2.00,0,NaN,NaN,NaN,NaN
Patient/3635188,10100295,1,1,101,1,2,2,1,11.9,3.0,151.0,23.4,20.10,0.20,9,0,0,11,0,0,1,0,4,35,36,35,48,207,48,207,14.74,3.20,1.305,0.31,1.43,1.58,97.0,65.0,125.0,10.0,0,1637.0,2.677607,3.41,303.000000,3.20,1,5.0,2.0,1.055556,NaN
Patient/3635189,10100297,1,2,101,1,2,2,1,11.7,6.0,132.2,20.9,23.29,0.46,9,1,0,11,0,0,0,0,2,207,28,207,2,207,48,207,23.01,-7.10,1.939,0.91,2.41,1.60,68.0,91.0,101.0,12.0,0,70.0,2.863917,1.40,205.357143,NaN,1,NaN,NaN,0.550000,180.0
Patient/3635190,10100300,1,1,101,1,2,2,3,8.5,1.5,146.7,29.5,17.07,2.20,4,0,0,2,0,0,0,1,2,9,20,69,20,207,28,207,23.49,-15.60,1.314,-0.46,1.93,1.57,88.0,117.0,123.0,24.0,0,176.0,3.018700,2.24,243.125000,0.40,1,3.0,NaN,0.100000,NaN
Patient/3635191,10100302,1,1,101,1,2,2,1,7.5,6.0,130.2,29.5,17.68,1.89,9,1,0,2,0,1,0,0,48,207,8,207,48,86,48,207,0.76,4.60,0.173,-1.20,1.55,1.30,73.0,90.0,125.0,1.0,0,70.0,2.778874,2.81,255.000000,0.60,1,NaN,NaN,0.400000,180.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Patient/3635995,10103334,1,1,101,1,2,2,3,10.0,4.0,147.7,37.4,17.11,1.32,8,1,0,12,0,1,0,0,4,104,48,207,2,207,48,207,19.13,3.59,0.463,1.31,2.48,3.55,64.0,112.0,103.0,25.0,0,583.0,2.477121,1.52,255.833333,4.25,1,6.0,4.0,0.166667,NaN
Patient/3635996,10103338,1,1,101,1,2,2,1,8.0,7.0,119.9,56.3,17.09,-0.34,9,0,0,11,0,1,1,0,20,9,12,207,48,86,48,207,8.99,NaN,0.381,-0.11,1.36,2.86,84.0,102.0,126.0,5.0,0,229.0,NaN,3.04,145.000000,NaN,1,3.0,NaN,0.923077,180.0
Patient/3635997,10103341,1,1,101,1,2,2,4,5.5,2.3,140.7,23.3,17.29,0.04,4,0,0,3,0,1,1,0,2,9,20,207,48,207,48,207,39.58,7.65,2.991,1.47,2.36,1.56,81.0,100.0,111.0,27.0,0,721.0,2.865104,2.63,135.000000,NaN,1,2.0,NaN,0.714286,180.0
Patient/3635998,10103346,1,1,101,1,2,2,1,5.7,2.5,121.3,26.2,16.59,0.96,4,0,0,3,0,0,0,1,4,52,48,207,48,207,48,207,17.36,-16.52,0.659,-0.12,1.78,1.21,89.0,82.0,NaN,15.0,0,422.0,3.126781,2.52,419.181818,3.30,0,1.0,1.0,0.777778,180.0


Each Observation will represent one cell in the above table.

813 rows × 51 columns = 41463

🖐 Note that is not the same total 41224 listed at the end of our queries. You may want to explore why. NaN is not the only reason for the difference.

Save the table above to a file

🖐 subtitute the filename below as you wish

In [10]:
txt_file_path = "results/phs002409_workaround_obs.txt"
patient_df.to_csv(txt_file_path, sep="\t")

### Query on observation value

#### Text value
Exercise to do - Formulate the query that would identify where the value of ENV_SMOKE is 'yes'

Ahead of doing that we can check by independent<sup>1</sup> means which patients we should get back from such a query.

The filter below on the DataFrame identfies 6 patients which match these criteria.

In [11]:
patient_df[patient_df.ENV_SMOKE == "1"]

,subj_id,consent_group,Gender,protocol,base_vnum,case_control,case_control_peak,Race - 2 character code PhenX,age_baseline,age_onset,Body height stand,Weight,Growth Chart Current BMI,bmiz_baseline,tx_grp,cs_tx,cs_comb_tx,clinic_city,smoke,ENV_SMOKE,ENV_SMOKE_pretrial,IUS,pred_bursts_event1_month,pred_bursts_event1_week,pred_bursts_event2_month,pred_bursts_event2_week,edhos_event1_month,edhos_event1_week,edhos_event2_month,edhos_event2_week,prech_long,prech_short,PC20_baseline,LNPC20_baseline,FEV1,FVC Vol Respiratory Spirometry,FEV1/FVC Predicted,PREFEVPP_baseline,PREFVCPP_baseline,bdrabpct_baseline,ATOPY,CompleteBloodCount Eosinophils Number,log10eos_baseline,IgE Concentration,ampf_baseline,totres_baseline,POS_SKINTEST_baseline,tot_pred_bursts,tot_edhos,amsym_baseline,cs_dose
Patient/3635191,10100302,1,1,101,1,2,2,1,7.5,6.0,130.2,29.5,17.68,1.89,9,1,0,2,0,1,0,0,48,207,8,207,48,86,48,207,0.76,4.60,0.173,-1.20,1.55,1.30,73.0,90.0,125.0,1.0,0,70.0,2.778874,2.81,255.000000,0.600000,1,NaN,NaN,0.400000,180.0
Patient/3635195,10100312,1,1,101,1,2,2,2,7.9,7.6,142.9,19.1,21.99,0.79,9,0,0,1,0,1,0,0,8,207,48,17,48,207,48,207,26.69,9.26,6.282,2.23,1.77,2.00,60.0,84.0,100.0,1.0,0,264.0,2.303196,2.25,252.272727,1.833333,1,1.0,NaN,0.100000,NaN
Patient/3635197,10100317,1,2,101,1,2,2,1,10.9,4.5,130.5,27.1,14.05,1.70,4,1,0,3,0,1,0,0,8,17,28,52,8,138,48,207,25.25,3.16,0.430,-0.53,1.09,1.84,87.0,98.0,111.0,6.0,1,NaN,2.699838,2.93,230.000000,0.230769,1,3.0,NaN,0.058824,NaN
Patient/3635198,10100319,1,2,101,1,2,2,1,12.1,2.0,118.2,43.0,22.42,-0.50,4,0,0,2,0,1,0,0,48,9,48,104,32,35,48,207,11.20,-8.75,12.074,1.75,1.17,2.76,77.0,59.0,94.0,10.0,0,141.0,NaN,3.09,246.666667,5.411765,1,9.0,2.0,0.379310,NaN
Patient/3635200,10100322,1,1,101,1,2,2,1,10.7,0.3,119.4,31.6,14.07,-1.07,9,1,0,10,0,1,0,0,4,207,48,52,48,207,48,207,14.86,-5.30,1.420,-2.25,1.22,2.30,87.0,106.0,106.0,9.0,1,563.0,2.892651,0.60,214.333333,0.266667,1,4.0,NaN,1.272727,180.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Patient/3635993,10103329,1,1,101,1,2,2,1,5.8,1.0,124.3,30.0,18.53,-0.16,4,1,0,2,0,1,1,0,2,9,36,35,48,207,48,207,6.90,13.76,0.719,0.80,1.89,1.12,65.0,99.0,103.0,6.0,0,105.0,3.109579,3.10,221.833333,4.133333,1,4.0,NaN,NaN,NaN
Patient/3635994,10103333,1,2,101,1,2,2,1,10.5,0.1,127.4,31.0,18.26,-0.07,4,0,0,8,0,1,1,1,8,207,48,35,2,207,48,207,28.44,10.10,1.518,-1.20,1.03,2.71,80.0,112.0,109.0,16.0,0,880.0,2.903633,1.28,155.666667,4.166667,1,10.0,NaN,0.500000,180.0
Patient/3635995,10103334,1,1,101,1,2,2,3,10.0,4.0,147.7,37.4,17.11,1.32,8,1,0,12,0,1,0,0,4,104,48,207,2,207,48,207,19.13,3.59,0.463,1.31,2.48,3.55,64.0,112.0,103.0,25.0,0,583.0,2.477121,1.52,255.833333,4.250000,1,6.0,4.0,0.166667,NaN
Patient/3635996,10103338,1,1,101,1,2,2,1,8.0,7.0,119.9,56.3,17.09,-0.34,9,0,0,11,0,1,1,0,20,9,12,207,48,86,48,207,8.99,NaN,0.381,-0.11,1.36,2.86,84.0,102.0,126.0,5.0,0,229.0,NaN,3.04,145.000000,NaN,1,3.0,NaN,0.923077,180.0


#### Numeric value
Exercise to do - write the FHIR query that would identify patients from this study where the PREFEVPP_baseline is greater than 120.

Again we can use our DataFrame for an independent<sup>1</sup> test to indicate which Patients we would expect to result from such a FHIR query

In [12]:
patient_df[patient_df.PREFEVPP_baseline > 120]

,subj_id,consent_group,Gender,protocol,base_vnum,case_control,case_control_peak,Race - 2 character code PhenX,age_baseline,age_onset,Body height stand,Weight,Growth Chart Current BMI,bmiz_baseline,tx_grp,cs_tx,cs_comb_tx,clinic_city,smoke,ENV_SMOKE,ENV_SMOKE_pretrial,IUS,pred_bursts_event1_month,pred_bursts_event1_week,pred_bursts_event2_month,pred_bursts_event2_week,edhos_event1_month,edhos_event1_week,edhos_event2_month,edhos_event2_week,prech_long,prech_short,PC20_baseline,LNPC20_baseline,FEV1,FVC Vol Respiratory Spirometry,FEV1/FVC Predicted,PREFEVPP_baseline,PREFVCPP_baseline,bdrabpct_baseline,ATOPY,CompleteBloodCount Eosinophils Number,log10eos_baseline,IgE Concentration,ampf_baseline,totres_baseline,POS_SKINTEST_baseline,tot_pred_bursts,tot_edhos,amsym_baseline,cs_dose
Patient/3635331,10100679,1,1,101,1,2,2,1,7.6,1.0,131.5,41.7,15.17,0.94,9,1,0,3,0,1,1,0,8,207,48,207,48,9,48,207,97.62,5.00,0.497,1.28,NaN,2.01,64.0,122.0,121.0,19.0,0,160.0,2.944976,2.13,202.083333,3.111111,0,NaN,1.0,1.107143,NaN
Patient/3635344,10100715,1,2,101,1,2,2,3,9.3,2.0,160.3,43.0,20.90,-0.79,9,0,0,8,0,1,0,0,2,86,48,207,48,207,48,207,18.18,9.49,0.526,-0.90,1.13,1.88,90.0,122.0,108.0,NaN,0,563.0,2.068186,2.96,277.750000,NaN,1,4.0,NaN,1.000000,180.0
Patient/3635385,10100810,1,1,101,1,2,2,1,10.2,5.0,157.8,39.2,21.48,-0.14,9,0,0,11,0,1,1,1,16,9,16,86,8,207,48,207,20.85,1.89,0.178,0.39,0.81,2.19,83.0,122.0,85.0,15.0,0,1100.0,2.778874,2.25,258.500000,2.444444,1,8.0,NaN,0.133333,180.0
Patient/3635403,10100873,1,1,101,1,2,2,1,9.0,3.5,141.2,63.6,17.22,-3.08,9,0,0,11,0,0,0,0,48,35,8,207,8,207,48,207,12.17,18.32,1.413,2.59,1.59,2.95,79.0,121.0,104.0,7.0,0,462.0,2.592177,1.89,178.076923,3.000000,1,NaN,NaN,NaN,NaN
Patient/3635419,10100913,1,1,101,1,2,2,1,9.6,2.0,116.9,30.9,15.66,0.53,8,0,0,8,0,0,1,0,2,156,32,207,2,190,40,207,32.06,26.74,1.470,-0.07,2.83,1.94,85.0,126.0,98.0,2.0,0,300.0,2.935003,157.00,251.551724,2.307692,1,1.0,NaN,1.000000,180.0
Patient/3635421,10100918,1,1,101,1,2,2,1,10.6,5.0,120.2,51.0,16.92,1.98,9,1,0,2,0,0,0,0,12,35,12,17,48,207,12,207,10.14,4.30,2.458,-0.31,1.44,2.05,72.0,148.0,128.0,5.0,0,128.0,2.777427,2.90,274.125000,4.181818,1,9.0,2.0,NaN,NaN
Patient/3635431,10100948,1,1,101,1,2,2,1,11.8,8.0,155.1,27.5,18.36,1.98,9,0,0,3,0,1,0,0,8,17,16,122,48,104,48,207,9.56,32.48,1.023,1.16,1.18,1.43,78.0,129.0,104.0,28.0,0,710.0,2.696356,1.93,161.153846,0.714286,0,3.0,NaN,0.320000,NaN
Patient/3635437,10100962,1,1,101,1,2,2,2,6.4,7.0,121.3,25.7,14.19,2.08,8,1,0,12,0,1,1,0,4,69,24,207,32,207,48,207,13.14,13.27,7.179,-0.77,1.60,1.49,75.0,124.0,97.0,27.0,0,640.0,2.751279,3.13,292.777778,5.896552,1,NaN,2.0,0.500000,NaN
Patient/3635521,10101193,1,1,101,1,2,2,1,11.1,3.0,139.9,48.5,16.51,-0.54,8,0,0,10,0,1,0,0,4,17,8,207,48,207,48,207,16.96,5.46,0.309,0.01,1.91,1.70,94.0,127.0,124.0,2.0,0,590.0,2.644439,2.95,269.000000,3.600000,1,NaN,1.0,NaN,180.0
Patient/3635564,10101289,1,2,101,1,2,2,1,9.4,0.3,107.9,26.6,15.13,0.90,9,0,0,11,0,0,0,0,4,207,12,207,48,207,32,207,21.48,3.53,3.040,-0.45,2.10,2.27,84.0,121.0,99.0,1.0,0,546.0,2.953760,3.74,295.294118,3.454545,1,3.0,NaN,0.233333,180.0


<sup>1</sup> Neither of the tests above is a completely indpendent test, as the DataFrame was itself generated via the FHIR API. A better test would rely on an independent means of accessing the source data.

### Queries on specific Observation values

In [13]:
vals = mf.run_query(
    "Observation?combo-code-value-quantity=phv00492057.v1.p1$gt1"
)

Total  Resources: 803
Total  Bytes: 1923140
Total  Pages: 9
Time elapsed 0.8773 seconds


In [14]:
vals = mf.run_query(
    "Observation?combo-code-value-quantity=PX091601370000$gt150"
)

Total  Resources: 106
Total  Bytes: 361735
Total  Pages: 2
Time elapsed 0.2176 seconds
